# Selección y Model Registry

| | |
|---|---|
| **Entrega** | PSet 3, sección 8 de `CD_PSet_3.pdf` |
| **Código** | `src/seleccion.py` |
| **Hold-out** | se abre **una sola vez**: el resultado queda en `docs/holdout_finalistas.csv` y en corridas `final` de MLflow; si ya existe, el notebook lo lee y no vuelve a evaluar |

**Sección 8 del PDF.** Elige el ganador con evidencia en los experimentos:
1. Compara el ganador con el segundo mejor usando el promedio y la desviación de la CV. ¿La diferencia supera la
   variación entre pliegues?
2. Evalúa ambos una sola vez en el conjunto de prueba.
3. Calcula la métrica de negocio.
4. Compara el costo y la explicabilidad: tiempo de entrenamiento e inferencia. ¿Puedes explicar por qué el modelo hace
   una predicción?

Registra el ganador en el Model Registry de MLflow (`<proyecto>-modelo`, alias `champion` y `challenger`), recárgalo en
un notebook limpio (`06_champion.ipynb`) y exporta las corridas a `docs/corridas.csv`.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.datos import cargar
from src.pipeline import preparar, dividir, FEATURES, TARGET, pliegues
from src.negocio import RiesgoDiario, portafolio_universo, serie_indice, resumen
from src import seleccion
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.grid': True})
obt, metadata = cargar()
datos = preparar(obt)
train, holdout = dividir(datos)
dia, metadata_dia = cargar(grano='dia')
riesgo = RiesgoDiario(dia)
print({'fuente': metadata['tabla'], 'sha256': metadata['sha256'], 'filas_obt': len(obt), 'entrenamiento': len(train), 'holdout_reservado': len(holdout)})

## 8.1 Finalistas por la métrica técnica (MAE de CV)

Se juntan los candidatos de la sección 6 y la mejor configuración de cada tuning (sección 7), y se ordenan por el MAE
de CV. Las líneas base no compiten: son la vara.

In [ ]:
f = seleccion.finalistas()
display(f.drop(columns=['model_uri'], errors='ignore').round(4))
registrables = f[f.registrable]
CHAMPION, CHALLENGER = registrables.iloc[0], registrables.iloc[1]
fuera = f.loc[:registrables.index[1]][~f.loc[:registrables.index[1]].registrable]
if len(fuera):
    print('Aviso: entre los dos primeros hay modelos sin Pipeline registrable (fundacionales en zero-shot):',
          fuera.modelo.tolist(), '-> se reportan, pero champion y challenger deben poder cargarse desde el Model Registry.')
print(f'champion: {CHAMPION.modelo} ({CHAMPION.version}), MAE CV {CHAMPION.mae_cv_media:.4f} ± {CHAMPION.mae_cv_std:.4f}')
print(f'challenger: {CHALLENGER.modelo} ({CHALLENGER.version}), MAE CV {CHALLENGER.mae_cv_media:.4f} ± {CHALLENGER.mae_cv_std:.4f}')

## 8.2 ¿La diferencia supera la variación entre pliegues?

In [ ]:
comp = seleccion.comparar_pliegues(CHAMPION.run_id, CHALLENGER.run_id)
display(pd.Series(comp).to_frame('champion vs challenger'))
if comp.get('pliegues'):
    print('La diferencia media de MAE (%.4f) %s la desviación entre pliegues del champion (%.4f); gana en %d de %d años.'
          % (comp['diferencia_media'], 'SUPERA' if comp['supera_variacion'] else 'NO supera',
             comp['desv_entre_pliegues_a'], comp['pliegues_que_gana_a'], comp['pliegues']))

## 8.3 Hold-out: una sola evaluación

In [ ]:
RUTA_HO = ROOT / 'docs/holdout_finalistas.csv'
if RUTA_HO.exists():
    ho = pd.read_csv(RUTA_HO)
    print('El hold-out ya se evaluó (', ho.run_id.tolist(), '): se leen los resultados, no se vuelve a evaluar.')
else:
    filas = [seleccion.evaluar_holdout(CHAMPION.modelo, CHAMPION.model_uri, holdout, metadata, 'champion', riesgo, metadata_dia, CHAMPION.run_id),
             seleccion.evaluar_holdout(CHALLENGER.modelo, CHALLENGER.model_uri, holdout, metadata, 'challenger', riesgo, metadata_dia, CHALLENGER.run_id)]
    ho = pd.DataFrame(filas); ho.to_csv(RUTA_HO, index=False)
cols = ['modelo', 'rol', 'mae_holdout', 'rmse_holdout', 'r2_vs_rw_holdout', 'acierto_direccional_holdout', 'ic_media_holdout',
        'negocio_sharpe_neto_holdout', 'negocio_drawdown_max_holdout', 'negocio_retorno_anual_neto_holdout', 'ms_inferencia_por_fila', 'run_id']
display(ho[[x for x in cols if x in ho]].round(4))

In [ ]:
# Referencias en los mismos meses del hold-out (no son modelos: B1 con la media del entrenamiento, B5 y el S&P 500).
from src.pipeline import metricas_regresion
b1 = metricas_regresion(holdout[TARGET], np.full(len(holdout), train[TARGET].mean()))
refs = {'B1 media (MAE)': b1['mae'],
        'B5 universo S&P 500 ∩ NYSE (Sharpe neto)': resumen(portafolio_universo(holdout))['sharpe_neto'],
        'S&P 500 índice de precio (Sharpe)': resumen(serie_indice(holdout, 'ret_log_sp500_mes_siguiente'))['sharpe_neto']}
display(pd.Series(refs).round(4).to_frame('hold-out 2024-01..2026-10'))

## 8.4 Costo y explicabilidad

In [ ]:
import mlflow.sklearn
filas = []
for fila, rol in ((CHAMPION, 'champion'), (CHALLENGER, 'challenger')):
    m = mlflow.sklearn.load_model(fila.model_uri)
    est = m.steps[-1][1]
    nombres = m[:-1].get_feature_names_out() if hasattr(m[:-1], 'get_feature_names_out') else None
    if hasattr(est, 'coef_'):
        peso, tipo = np.ravel(est.coef_), 'coeficiente (variables escaladas)'
    elif hasattr(est, 'feature_importances_'):
        peso, tipo = np.asarray(est.feature_importances_, float), 'importancia (ganancia)'
    else:
        peso, tipo = None, 'sin explicación directa'
    if peso is not None and nombres is not None:
        top = pd.Series(peso, index=nombres).sort_values(key=np.abs, ascending=False).head(8).round(4)
        print(f'{rol} {fila.modelo} - {tipo}:'); display(top.to_frame(tipo))
    filas.append({'rol': rol, 'modelo': fila.modelo, 'segundos_entrenamiento_cv': fila.segundos_entrenamiento,
                  'ms_inferencia_por_fila_holdout': float(ho.set_index('rol').loc[rol, 'ms_inferencia_por_fila']),
                  'explicacion': tipo})
display(pd.DataFrame(filas).round(4))

## 8.5 Model Registry y exportación de corridas

In [ ]:
from mlflow import MlflowClient
cliente = MlflowClient()
try:
    alias = {a: cliente.get_model_version_by_alias(seleccion.NOMBRE_MODELO, a).version for a in ('champion', 'challenger')}
    print('Ya registrado:', alias)
except Exception:
    print(seleccion.registrar_en_registry(CHAMPION.model_uri, CHALLENGER.model_uri))
print('Corridas exportadas a', seleccion.exportar_corridas())

**Decisión.** *(Se completa con los resultados: ganador, diferencia frente al segundo y frente a la variación entre
pliegues, desempeño en el hold-out frente a B1 y B5, costo y explicabilidad.)*